# Análise Exploratória dos Dados

## Dicionário de dados

In [1]:
# importar os pacotes necessários
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dicionario = pd.read_csv("../data/raw/dicionario_dados.csv", sep=";")
dicionario

,base,coluna,descricao
0,base_cadastral,id_cliente,Identificador único do cliente
1,base_cadastral,sexo,Sexo do cliente (M ou F)
2,base_cadastral,data_nascimento,Data de nascimento do cliente
3,base_cadastral,qtd_filhos,Quantidade de filhos declarados
4,base_cadastral,qtd_membros_familia,Quantidade total de membros da família
5,base_cadastral,renda_anual,Renda total anual declarada pelo cliente
6,base_cadastral,tipo_renda,"Tipo de fonte de renda (trabalho, pensão, apos..."
7,base_cadastral,ocupacao,Ocupação profissional do cliente
8,base_cadastral,tipo_organizacao,Tipo de empresa onde o cliente trabalha
9,base_cadastral,nivel_educacao,Nível de escolaridade alcançado


## Funções

## Criação das Métricas Históricas

In [3]:
# importar as tabelas
cadastral = pd.read_parquet('../data/raw/base_cadastral.parquet', engine="fastparquet")
submissao = pd.read_parquet('../data/raw/base_submissao.parquet', engine="fastparquet") 
hist_contratos = pd.read_parquet('../data/raw/historico_emprestimos.parquet', engine="fastparquet")
hist_parcelas = pd.read_parquet('../data/raw/historico_parcelas.parquet', engine="fastparquet")

In [4]:
submissao.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5
3,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5
4,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5


In [5]:
cadastral.head()

,id_cliente,sexo,data_nascimento,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
0,100023,F,1994-01-30,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
1,100031,F,1973-11-13,0,1.0,112500.0,Working,Cooking staff,Business Entity Type 3,Secondary / secondary special,Widow,House / apartment,N,Y,3,2
2,100056,M,1975-02-19,0,2.0,360000.0,Working,Laborers,Transport: type 2,Secondary / secondary special,Married,House / apartment,Y,Y,2,2
3,100069,M,1986-04-10,1,2.0,360000.0,Working,Laborers,Transport: type 4,Secondary / secondary special,Separated,House / apartment,Y,Y,2,2
4,100085,M,1994-07-05,1,3.0,157500.0,Working,Drivers,Business Entity Type 1,Secondary / secondary special,Married,House / apartment,N,Y,2,2


In [6]:
cadastral.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_cliente                  40000 non-null  int64  
 1   sexo                        40000 non-null  object 
 2   data_nascimento             40000 non-null  object 
 3   qtd_filhos                  40000 non-null  int64  
 4   qtd_membros_familia         40000 non-null  float64
 5   renda_anual                 40000 non-null  float64
 6   tipo_renda                  40000 non-null  object 
 7   ocupacao                    27324 non-null  object 
 8   tipo_organizacao            40000 non-null  object 
 9   nivel_educacao              40000 non-null  object 
 10  estado_civil                40000 non-null  object 
 11  tipo_moradia                40000 non-null  object 
 12  possui_carro                40000 non-null  object 
 13  possui_imovel               40000 non-null

In [ ]:
# unir historico de contratos com parcelas
hist_total = hist_contratos.merge(
    hist_parcelas, 
    how="left", 
    on=["id_contrato", "id_cliente"]
)

hist_total["dias_atraso_parcela"] = (pd.to_datetime(hist_total["data_real_pagamento"]) - pd.to_datetime(hist_total["data_prevista_pagamento"])).dt.days

# CRIACAO DA VARIÁVEL ALVO DE ACORDO COM O REGULADOR (BACEN)
hist_total["target"] = np.where(
    hist_total["dias_atraso_parcela"] > 90,
    1,
    0
)

features_temp = hist_total.groupby(
    [
        "id_cliente",
        "id_contrato", 
        "tipo_contrato", 
        "status_contrato", 
        "data_decisao", 
        "valor_credito", 
        "valor_bem", 
        "percentual_entrada",
        "qtd_parcelas_planejadas",
        "tipo_pagamento",
        "finalidade_emprestimo",
        "tipo_cliente",
        "faixa_rendimento",
        "tipo_portfolio",
        "tipo_produto",
        "categoria_bem",
        "combinacao_produto",
        "setor_vendedor",
        "dia_semana_solicitacao",
        "hora_solicitacao",
        "flag_ultima_solicitacao_contrato",
        "flag_ultima_solicitacao_dia",
        "motivo_recusa",
        "acompanhantes_cliente",
        "flag_seguro_contratado"
    ]
).agg(
num_max_parcelas=("numero_parcela", "max"),
# num_max_atraso_contrato=("dias_atraso_parcela", "max"),
valor_total_a_pagar=("valor_parcela", "sum"),
valor_medio_pago_contrato=("valor_pago_parcela", "mean"),
valor_max_pago_contrato=("valor_pago_parcela", "max"),
target=("target", "max"),
).reset_index()

features_temp = features_temp.sort_values(by=["id_cliente", "data_decisao"])

In [13]:
features_temp.loc[features_temp["data_decisao"] > "2025-01-01"].head()

,id_cliente,id_contrato,tipo_contrato,status_contrato,data_decisao,valor_credito,valor_bem,percentual_entrada,qtd_parcelas_planejadas,tipo_pagamento,finalidade_emprestimo,tipo_cliente,faixa_rendimento,tipo_portfolio,tipo_produto,categoria_bem,combinacao_produto,setor_vendedor,dia_semana_solicitacao,hora_solicitacao,flag_ultima_solicitacao_contrato,flag_ultima_solicitacao_dia,motivo_recusa,acompanhantes_cliente,flag_seguro_contratado,qtd_contratos,num_max_parcelas,valor_total_a_pagar,valor_medio_pago_contrato,valor_max_pago_contrato,target
31,100278,1770201,Consumer loans,Approved,2025-01-03,133038.0,130455.0,0.100334,10.0,XNA,XAP,New,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,FRIDAY,11,Y,1,XAP,Unaccompanied,1.0,1,NaN,15654.690,NaN,NaN,0
72,100471,1153218,Consumer loans,Approved,2025-01-22,202455.0,202455.0,0.000000,18.0,XNA,XAP,Refreshed,middle,POS,XNA,Furniture,POS industry with interest,Furniture,WEDNESDAY,16,Y,1,XAP,Unaccompanied,0.0,1,NaN,14607.135,NaN,NaN,0
172,101205,2628025,Consumer loans,Approved,2025-02-07,36787.5,31756.5,0.000000,12.0,Cash through the bank,XAP,Repeater,high,POS,XNA,Mobile,POS mobile with interest,Connectivity,FRIDAY,12,Y,1,XAP,"Spouse, partner",1.0,1,NaN,4603.500,NaN,NaN,0
231,101583,2173085,Consumer loans,Approved,2025-01-03,350559.0,350559.0,0.000000,6.0,Cash through the bank,XAP,Repeater,low_action,POS,XNA,Furniture,POS industry without interest,Furniture,FRIDAY,16,Y,1,XAP,Unaccompanied,0.0,1,1.0,61632.585,61632.585,61632.585,0
272,101942,1031150,Consumer loans,Approved,2025-02-06,48208.5,44671.5,0.000000,6.0,Cash through the bank,XAP,Repeater,middle,POS,XNA,Audio/Video,POS household with interest,Consumer electronics,THURSDAY,9,Y,1,XAP,Family,1.0,1,NaN,9045.225,NaN,NaN,0


In [18]:
a = features_temp[[
    "id_cliente",
    "data_decisao"
]]

In [19]:
teste = submissao.merge(a, how="left", on="id_cliente")

teste.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela,data_decisao
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5,2018-07-07
1,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5,2020-02-01
2,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5,NaN
3,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5,NaN
4,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5,2020-11-07


In [20]:
teste.info()

<class 'pandas.DataFrame'>
RangeIndex: 60904 entries, 0 to 60903
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_cliente              60904 non-null  int64  
 1   data_solicitacao        60904 non-null  object 
 2   dia_semana_solicitacao  60904 non-null  object 
 3   hora_solicitacao        60904 non-null  int64  
 4   tipo_contrato           60904 non-null  object 
 5   valor_credito           60904 non-null  float64
 6   valor_bem               60867 non-null  float64
 7   valor_parcela           60895 non-null  float64
 8   data_decisao            48223 non-null  str    
dtypes: float64(3), int64(2), object(3), str(1)
memory usage: 4.2+ MB


In [23]:
teste.loc[
    teste["data_decisao"] == teste.groupby("id_cliente")["data_decisao"].transform("max")
].head(50)

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela,data_decisao
1,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5,2020-02-01
7,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5,2023-03-26
8,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5,2021-12-02
9,100098,2025-02-05,WEDNESDAY,9,Revolving loans,270000.0,270000.0,13500.0,2017-11-28
10,100104,2025-02-19,WEDNESDAY,17,Cash loans,547344.0,472500.0,30690.0,2018-01-16
14,100111,2025-02-17,MONDAY,11,Cash loans,862560.0,720000.0,27954.0,2023-06-07
15,100123,2025-02-15,SATURDAY,16,Cash loans,675000.0,675000.0,19737.0,2020-07-02
17,100165,2025-02-04,TUESDAY,16,Cash loans,1293502.5,1129500.0,35568.0,2020-11-03
21,100208,2025-02-21,FRIDAY,11,Cash loans,888840.0,675000.0,29016.0,2022-01-21
24,100233,2025-02-05,WEDNESDAY,10,Cash loans,679671.0,607500.0,28926.0,2024-12-16
